In [1]:
import pandas as pd
import asyncio
import aiohttp
from pathlib import Path
from tqdm.asyncio import tqdm
import re 

In [2]:
DATA_DIR = Path.cwd().parent / "data"
ITEMS_PATH = DATA_DIR / "items.csv"
OUTPUT_PATH = DATA_DIR / "enriched_items.csv"

In [4]:
async def fetch_book_data(session, item_id, isbn, semaphore):
    """
    Fetches book metadata from OpenLibrary using the ISBN.
    Uses a semaphore to prevent overwhelming the API.
    """
    if pd.isna(isbn) or str(isbn).strip() == "":
        return {"item_id": item_id, "description": "", "pages": None, "api_found": False}

    # OpenLibrary API endpoint using ISBN
    url = f"https://openlibrary.org/api/books?bibkeys=ISBN:{isbn}&format=json&jscmd=data"
    
    async with semaphore:
        try:
            async with session.get(url, timeout=10) as response:
                if response.status == 200:
                    data = await response.json()
                    book_key = f"ISBN:{isbn}"
                    
                    if book_key in data:
                        book_info = data[book_key]
                        
                        # OpenLibrary sometimes stores descriptions as text, sometimes it's missing from this endpoint.
                        # We grab whatever useful text context we can (e.g., subjects, notes).
                        subjects = [s['name'] for s in book_info.get('subjects', [])]
                        subject_text = ", ".join(subjects)
                        
                        return {
                            "item_id": item_id,
                            "description": book_info.get('notes', subject_text), # Fallback to subjects if no notes
                            "pages": book_info.get('number_of_pages', None),
                            "api_found": True
                        }
                
                # Small sleep to respect rate limits
                await asyncio.sleep(0.1)
                
        except Exception as e:
            # Catch timeouts or connection errors silently to keep the loop running
            pass
            
    return {"item_id": item_id, "description": "", "pages": None, "api_found": False}

async def main():
    # 1. Load the original items
    print("Loading items.csv")
    items_df = pd.read_csv(ITEMS_PATH)
    
    # We will limit concurrency to 10 to avoid getting temporarily banned by OpenLibrary
    semaphore = asyncio.Semaphore(10)
    
    # 2. Set up the async session and tasks
    async with aiohttp.ClientSession() as session:
        tasks = []
        for _, row in items_df.iterrows():
            # Adjust column names based on your actual items.csv header (e.g., 'item_id', 'ISBN')
            task = fetch_book_data(session, row['i'], row['ISBN Valid'], semaphore) 
            tasks.append(task)
        
        print(f"Fetching metadata for {len(tasks)} books from OpenLibrary")
        
        # 3. Run tasks with a progress bar
        results = await tqdm.gather(*tasks)
    
    # 4. Merge results back into the original dataframe
    print("Processing results")
    enriched_data = pd.DataFrame(results)
    
    # Merge on the item ID (assuming your items.csv uses 'i' like the interactions df)
    final_df = items_df.merge(enriched_data, left_on='i', right_on='item_id', how='left')
    final_df.drop(columns=['item_id'], inplace=True)
    
    # 5. Save to CSV
    final_df.to_csv(OUTPUT_PATH, index=False)
    print(f"Success! Enriched data saved to {OUTPUT_PATH}")
    
    # Print a quick summary of how many books we successfully found
    found_count = final_df['api_found'].sum()
    print(f"Found additional data for {found_count} out of {len(final_df)} books.")

# Run the async loop
await main()

Loading items.csv
Fetching metadata for 15291 books from OpenLibrary


100%|██████████| 15291/15291 [1:20:53<00:00,  3.15it/s]


Processing results
Success! Enriched data saved to c:\Users\Julien\OneDrive\VS Code\OneDrive\GitHub\library-recommender\data\enriched_items.csv
Found additional data for 97 out of 15291 books.


In [4]:
import re 
# Set up paths 
DATA_DIR = Path.cwd().parent / "data"
ITEMS_PATH = DATA_DIR / "items.csv"
OUTPUT_PATH = DATA_DIR / "enriched_items_v2.csv"

def extract_valid_isbns(raw_isbn_str):
    """
    Splits by semicolon, removes hyphens/spaces, and filters for 10 or 13 length codes.
    """
    if pd.isna(raw_isbn_str) or str(raw_isbn_str).strip() == "":
        return []
    
    raw_codes = str(raw_isbn_str).split(';')
    valid_isbns = []
    
    for code in raw_codes:
        # Strip out dashes and whitespace
        clean_code = re.sub(r'[\s-]', '', code)
        
        # A basic heuristic: ISBNs are generally 10 or 13 characters long.
        # (ISBN-10 can sometimes end in an 'X')
        if len(clean_code) in [10, 13]:
            valid_isbns.append(clean_code)
            
    return valid_isbns

async def fetch_book_data(session, item_id, isbns, semaphore):
    """
    Queries OpenLibrary using a list of multiple ISBNs.
    """
    # If no valid ISBNs were extracted, skip the API call
    if not isbns:
        return {"item_id": item_id, "description": "", "pages": None, "api_found": False}

    # OpenLibrary allows batching keys! Example: "ISBN:123,ISBN:456"
    bibkeys = ",".join([f"ISBN:{isbn}" for isbn in isbns])
    url = f"https://openlibrary.org/api/books?bibkeys={bibkeys}&format=json&jscmd=data"
    
    async with semaphore:
        try:
            async with session.get(url, timeout=10) as response:
                if response.status == 200:
                    data = await response.json()
                    
                    # Check which of the ISBNs returned a hit
                    for isbn in isbns:
                        book_key = f"ISBN:{isbn}"
                        if book_key in data:
                            book_info = data[book_key]
                            
                            subjects = [s['name'] for s in book_info.get('subjects', [])]
                            subject_text = ", ".join(subjects)
                            
                            return {
                                "item_id": item_id,
                                "description": book_info.get('notes', subject_text), 
                                "pages": book_info.get('number_of_pages', None),
                                "api_found": True
                            }
                
                # Small sleep to respect rate limits
                await asyncio.sleep(0.1)
                
        except Exception as e:
            pass
            
    return {"item_id": item_id, "description": "", "pages": None, "api_found": False}

async def main():
    print("Loading items.csv")
    items_df = pd.read_csv(ITEMS_PATH)
    
    semaphore = asyncio.Semaphore(10)
    
    async with aiohttp.ClientSession() as session:
        tasks = []
        for _, row in items_df.iterrows():
            # 1. Parse the ISBN column first
            isbns_list = extract_valid_isbns(row['ISBN Valid'])
            
            # 2. Pass the list to our API fetcher
            task = fetch_book_data(session, row['i'], isbns_list, semaphore) 
            tasks.append(task)
        
        print(f"Fetching metadata for {len(tasks)} books from OpenLibrary")
        results = await tqdm.gather(*tasks)
    
    print("Processing results")
    enriched_data = pd.DataFrame(results)
    
    final_df = items_df.merge(enriched_data, left_on='i', right_on='item_id', how='left')
    final_df.drop(columns=['item_id'], inplace=True)
    
    final_df.to_csv(OUTPUT_PATH, index=False)
    
    found_count = final_df['api_found'].sum()
    print(f"Success! Enriched data saved to {OUTPUT_PATH}")
    print(f"Found additional data for {found_count} out of {len(final_df)} books.")

await main()

Loading items.csv
Fetching metadata for 15291 books from OpenLibrary


100%|██████████| 15291/15291 [1:20:52<00:00,  3.15it/s]

Processing results
Success! Enriched data saved to c:\Users\Julien\OneDrive\VS Code\OneDrive\GitHub\library-recommender\data\enriched_items_v2.csv
Found additional data for 6371 out of 15291 books.


In [ ]:
import pandas as pd
import asyncio
import aiohttp
from pathlib import Path
from tqdm.asyncio import tqdm
import re
import urllib.parse
import os

DATA_DIR = Path.cwd().parent / "data"
ITEMS_PATH = DATA_DIR / "items.csv"
OUTPUT_PATH = DATA_DIR / "google_enriched_items.csv"

# Put multiple keys here. If you only have one, just leave one in the list.
# You can add more later if the script stops!
API_KEYS = []
current_key_idx = 0  # Tracks which key we are using

def get_current_key_param():
    """Returns the URL parameter for the active API key."""
    if not API_KEYS or API_KEYS[0] == "YOUR_KEY_1":
        return ""
    return f"&key={API_KEYS[current_key_idx]}"

def extract_valid_isbns(raw_isbn_str):
    if pd.isna(raw_isbn_str) or str(raw_isbn_str).strip() == "":
        return []
    raw_codes = str(raw_isbn_str).split(';')
    valid_isbns = [re.sub(r'[\s-]', '', code) for code in raw_codes]
    return [code for code in valid_isbns if len(code) in [10, 13]]

async def fetch_from_google(session, base_url):
    global current_key_idx
    for attempt in range(3):
        url = base_url + get_current_key_param()
        try:
            async with session.get(url, timeout=10) as response:
                if response.status == 200:
                    data = await response.json()
                    if "items" in data and len(data["items"]) > 0:
                        volume_info = data["items"][0].get("volumeInfo", {})
                        
                        genres = ", ".join(volume_info.get("categories", []))
                        description = volume_info.get("description", "")
                        
                        # NEW: Extract numerical features
                        page_count = volume_info.get("pageCount", None)
                        avg_rating = volume_info.get("averageRating", None)
                        
                        return genres, description, page_count, avg_rating
                    return None, None, None, None 
                
                elif response.status == 429:
                    current_key_idx = (current_key_idx + 1) % len(API_KEYS)
                    if current_key_idx == 0:
                        await asyncio.sleep(60)
        except Exception:
            pass
        await asyncio.sleep(1)
    return None, None, None, None

async def fetch_book_data(session, row, semaphore):
    item_id = row['i']
    
    # Extract values
    title = row.get('Title', '')
    author = row.get('Author', '')
    raw_isbn = row.get('ISBN Valid', '')
    
    isbns = extract_valid_isbns(raw_isbn)
    genres, description = None, None

    async with semaphore:
        # ---------------------------------------------------------
        # ATTEMPT 1: Search by ISBN
        # ---------------------------------------------------------
        for isbn in isbns:
            url = f"https://www.googleapis.com/books/v1/volumes?q=isbn:{isbn}"
            genres, description, page_count, avg_rating = await fetch_from_google(session, url)
            if genres or description:
                break 
        
        # ---------------------------------------------------------
        # ATTEMPT 2: Strict Title + Author (No Publisher)
        # ---------------------------------------------------------
        if not genres and not description and pd.notna(title) and str(title).strip():
            
            # Basic text cleaning to remove special characters that break the API
            clean_title = re.sub(r'[^\w\s]', '', str(title).strip())
            clean_title_url = urllib.parse.quote(clean_title)
            
            query_strict = f"intitle:{clean_title_url}"
            
            if pd.notna(author) and str(author).strip():
                clean_author = re.sub(r'[^\w\s]', '', str(author).strip())
                clean_author_url = urllib.parse.quote(clean_author)
                query_strict += f"+inauthor:{clean_author_url}"
                
            url_strict = f"https://www.googleapis.com/books/v1/volumes?q={query_strict}"
            genres, description, page_count, avg_rating = await fetch_from_google(session, url_strict)
            
            # ---------------------------------------------------------
            # ATTEMPT 3: Loose Keyword Search (Mimics Web Search)
            # ---------------------------------------------------------
            if not genres and not description:
                # Just mash the title and author together as general keywords
                query_loose = clean_title_url
                if pd.notna(author) and str(author).strip():
                    query_loose += f"+{clean_author_url}"
                
                url_loose = f"https://www.googleapis.com/books/v1/volumes?q={query_loose}"
                genres, description, page_count, avg_rating = await fetch_from_google(session, url_loose)
        
        await asyncio.sleep(0.15)

    return {
        "item_id": item_id,
        "genres": genres if genres else "",
        "summary": description if description else "",
        "page_count": page_count,
        "average_rating": avg_rating,
        "api_found": bool(genres or description)
    }

async def main():
    print("Loading items.csv")
    items_df = pd.read_csv(ITEMS_PATH)
    
    # CHECKPOINTING: Read existing data so we don't start from scratch
    if os.path.exists(OUTPUT_PATH):
        enriched_data = pd.read_csv(OUTPUT_PATH)
        completed_ids = set(enriched_data['item_id'].values)
        print(f"Found existing save file! {len(completed_ids)} books already fetched.")
    else:
        completed_ids = set()
        # Create an empty CSV with headers to start appending to
        pd.DataFrame(columns=["item_id", "genres", "summary", "api_found"]).to_csv(OUTPUT_PATH, index=False)
        enriched_data = pd.DataFrame()

    # Filter out items we already have
    items_to_fetch = items_df[~items_df['i'].isin(completed_ids)]
    print(f"Remaining books to fetch: {len(items_to_fetch)}")

    if len(items_to_fetch) == 0:
        print("All books have been fetched! You are ready to move on.")
        return

    semaphore = asyncio.Semaphore(5)
    
    # BATCH PROCESSING: Save progress every 200 items
    batch_size = 100
    
    async with aiohttp.ClientSession() as session:
        for i in range(0, len(items_to_fetch), batch_size):
            batch = items_to_fetch.iloc[i : i + batch_size]
            
            print(f"\nProcessing batch {i} to {i + len(batch)}")
            tasks = [fetch_book_data(session, row, semaphore) for _, row in batch.iterrows()]
            
            # Run the batch
            results = await tqdm.gather(*tasks)
            
            # Save the batch immediately to the CSV
            batch_df = pd.DataFrame(results)
            batch_df.to_csv(OUTPUT_PATH, mode='a', header=False, index=False)
            
            # Update our completed IDs in memory just in case
            completed_ids.update(batch_df['item_id'].values)

    print("\nFetch complete! Merging enriched data with original items")
    
    # Final merge just to give you a clean dataframe at the end
    full_enriched_data = pd.read_csv(OUTPUT_PATH)
    final_df = items_df.merge(full_enriched_data, left_on='i', right_on='item_id', how='left')
    final_df.drop(columns=['item_id'], inplace=True)
    
    final_csv_path = DATA_DIR / "final_enriched_items.csv"
    final_df.to_csv(final_csv_path, index=False)
    
    print(f"Success! Merged data saved to {final_csv_path}")
    print(f"Total APIs hits found: {final_df['api_found'].sum()}/{len(final_df)}")

await main()

In [ ]:
import requests
import urllib.parse
import json

# Insert your single API key here
API_KEY = ""

def test_google_books_search(search_type, search_value):
    """
    Tests the Google Books API for a single parameter.
    
    :param search_type: Must be 'isbn', 'title', or 'author'
    :param search_value: The value you are searching for (e.g., '9780441172719' or 'Dune')
    """
    # 1. Clean and format the search value for the URL
    clean_value = urllib.parse.quote(str(search_value).strip())
    
    # 2. Build the specific query parameter
    if search_type.lower() == 'isbn':
        query = f"isbn:{clean_value}"
    elif search_type.lower() == 'title':
        query = f"intitle:{clean_value}"
    elif search_type.lower() == 'author':
        query = f"inauthor:{clean_value}"
    else:
        print("Error: search_type must be 'isbn', 'title', or 'author'")
        return

    # 3. Construct the full URL
    url = f"https://www.googleapis.com/books/v1/volumes?q={query}&key={API_KEY}"
    print(f"Searching {search_type.upper()} for: '{search_value}'...")
    print(f"URL Generated: {url.replace(API_KEY, 'HIDDEN_API_KEY')}\n") # Hide key in console

    # 4. Make the request
    try:
        response = requests.get(url, timeout=10)
        
        # Check if the API key is valid / quota isn't exceeded
        if response.status_code != 200:
            print(f"API Error {response.status_code}: {response.text}")
            return
            
        data = response.json()
        
        # 5. Parse the results
        if "items" in data and len(data["items"]) > 0:
            # We just look at the top hit for testing
            top_hit = data["items"][0]
            volume_info = top_hit.get("volumeInfo", {})
            
            found_title = volume_info.get("title", "No Title Found")
            found_authors = ", ".join(volume_info.get("authors", ["No Authors Found"]))
            genres = ", ".join(volume_info.get("categories", ["No Genres Found"]))
            description = volume_info.get("description", "No Description Found")
            
            print("--- SUCCESS: BOOK FOUND ---")
            print(f"Title:   {found_title}")
            print(f"Authors: {found_authors}")
            print(f"Genres:  {genres}")
            print(f"Summary: {description[:200]}...") # Print first 200 chars to avoid flooding terminal
            
            # Optional: Uncomment the line below if you want to see the ENTIRE raw JSON response
            # print("\nRaw JSON:")
            # print(json.dumps(volume_info, indent=2))
            
        else:
            print("--- NO RESULTS FOUND ---")
            print("The API returned a 200 OK status, but its database has no matches for this query.")

    except Exception as e:
        print(f"Connection Error: {e}")

# ==========================================
# Run your tests here!
# ==========================================

# Test 1: By ISBN
# test_google_books_search('isbn', '9782871303336') # Find the Title and Author, no genres or summary   (Book id: 0)
# test_google_books_search("isbn", "2871303339")  # Same

# Test 2: By Title
test_google_books_search('title', 'Classification décimale universelle : édition abrégée')

# Test 3: By Author
# test_google_books_search('author', 'Victor Hugo')

Searching TITLE for: 'Classification décimale universelle : édition abrégée'...
URL Generated: https://www.googleapis.com/books/v1/volumes?q=intitle:Classification%20d%C3%A9cimale%20universelle%20%3A%20%C3%A9dition%20abr%C3%A9g%C3%A9e&key=HIDDEN_API_KEY

API Error 429: {
  "error": {
    "code": 429,
    "message": "Quota exceeded for quota metric 'Queries' and limit 'Queries per day' of service 'books.googleapis.com' for consumer 'project_number:713531405839'.",
    "errors": [
      {
        "message": "Quota exceeded for quota metric 'Queries' and limit 'Queries per day' of service 'books.googleapis.com' for consumer 'project_number:713531405839'.",
        "domain": "global",
        "reason": "rateLimitExceeded"
      }
    ],
    "status": "RESOURCE_EXHAUSTED",
    "details": [
      {
        "@type": "type.googleapis.com/google.rpc.ErrorInfo",
        "reason": "RATE_LIMIT_EXCEEDED",
        "domain": "googleapis.com",
        "metadata": {
          "quota_limit_value": "1000